In [0]:
"""
01_machine_kpis.py

Streaming Machine KPIs.

Computes machine-level manufacturing KPIs from
Silver Operation Events.

Input:
    operation_events

Output:
    machine_kpis

Author:
Sumanth Vempalle

Version:
2.1.0
"""

import dlt

from pyspark.sql.functions import (
    avg,
    col,
    count,
    current_timestamp,
    max,
    min,
    sum,
    when,
)

# ============================================================
# Machine KPIs
# ============================================================

@dlt.table(
    name="machine_kpis",
    comment="Machine-level manufacturing KPIs.",
    table_properties={
        "quality": "gold",
        "pipelines.autoOptimize.managed": "true",
    },
)

@dlt.expect_or_drop(
    "valid_machine_id",
    "machine_id IS NOT NULL",
)

@dlt.expect(
    "positive_operations",
    "operations_completed > 0",
)

def machine_kpis():

    operations = (

        dlt.read_stream(
            "operation_events"
        )

        .withWatermark(
            "event_timestamp",
            "10 minutes",
        )

    )

    return (

        operations

        .groupBy(

            "plant_code",

            "hall_id",

            "line_id",

            "machine_id",

        )

        .agg(

            count("*").alias(
                "operations_completed"
            ),

            sum(

                when(
                    col("quality_result") == "PASS",
                    1,
                ).otherwise(0)

            ).alias(
                "passed_operations"
            ),

            sum(

                when(
                    col("quality_result") == "FAIL",
                    1,
                ).otherwise(0)

            ).alias(
                "failed_operations"
            ),

            avg(
                "actual_force_kn"
            ).alias(
                "average_force_kn"
            ),

            min(
                "actual_force_kn"
            ).alias(
                "minimum_force_kn"
            ),

            max(
                "actual_force_kn"
            ).alias(
                "maximum_force_kn"
            ),

            avg(
                "cycle_time_sec"
            ).alias(
                "average_cycle_time_sec"
            ),

        )

        .withColumns({

            "pass_rate": (
                col("passed_operations")
                / col("operations_completed")
            ) * 100,

            "generated_timestamp": current_timestamp()

        })

    )